# Lab 9 · Mapa causal de calidad de servicio y abandono

Notebook para pasar de correlaciones a hipótesis causales. Se calcula la matriz de correlación, se dibuja un DAG y se reflexiona sobre confusores y validación causal.

In [ ]:
# Instalación de dependencias básicas. En SageMaker normalmente ya están instaladas.
%pip -q install pandas numpy matplotlib networkx scikit-learn s3fs

import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Por defecto se leen los CSV incluidos en la carpeta local data/.
# Si prefieres leer desde S3, cambia USE_S3=True y ajusta S3_PREFIX.
USE_S3 = False
DATA_DIR = Path('data')
S3_PREFIX = 's3://TU_BUCKET/TU_CARPETA'  # ejemplo: s3://mi-bucket/graph

def data_path(filename):
    return f"{S3_PREFIX}/{filename}" if USE_S3 else str(DATA_DIR / filename)

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

import networkx as nx

## Parte A. Cargar dataset causal

In [ ]:
df = pd.read_csv(data_path('telco_causal_lab9.csv'))
print(df.shape)
display(df.head())

## Parte B. Diferencias de medias por churn y región

In [ ]:
medias_churn = df.groupby('churn').agg(
    clientes=('customer_id','count'),
    calidad=('calidad_tecnica','mean'),
    interrupciones=('interrupciones_30d','mean'),
    brecha=('brecha_velocidad_pct','mean'),
    reclamaciones=('reclamaciones_30d','mean'),
    insatisfaccion=('insatisfaccion','mean'),
    precio=('precio_mensual','mean')
).round(2)
display(medias_churn)

medias_region = df.groupby('region').agg(
    clientes=('customer_id','count'),
    calidad=('calidad_tecnica','mean'),
    interrupciones=('interrupciones_30d','mean'),
    reclamaciones=('reclamaciones_30d','mean'),
    insatisfaccion=('insatisfaccion','mean'),
    pct_churn=('churn','mean')
).sort_values('calidad', ascending=False)
medias_region['pct_churn'] = (medias_region['pct_churn']*100).round(1)
display(medias_region.round(2))
medias_churn.to_csv(OUTPUT_DIR/'lab09_medias_por_churn.csv')
medias_region.to_csv(OUTPUT_DIR/'lab09_medias_por_region.csv')

## Parte C. Correlaciones cuantitativas

In [ ]:
num = df.drop(columns=['customer_id','region'])
corr = num.corr().round(2)
display(corr[['churn']].sort_values('churn', ascending=False))

plt.figure(figsize=(8,6))
plt.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
plt.xticks(range(len(corr)), corr.columns, rotation=90)
plt.yticks(range(len(corr)), corr.columns)
plt.colorbar(label='correlación')
plt.title('Matriz de correlación')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'lab09_matriz_correlacion.png', dpi=150, bbox_inches='tight')
plt.show()
corr.to_csv(OUTPUT_DIR/'lab09_correlaciones.csv')

## Parte D. Construir el mapa causal hipotético

In [ ]:
D = nx.DiGraph()
relaciones = [
    ('inversion_red','calidad_tecnica'),
    ('calidad_tecnica','interrupciones_30d'),
    ('calidad_tecnica','brecha_velocidad_pct'),
    ('interrupciones_30d','reclamaciones_30d'),
    ('interrupciones_30d','tiempo_resolucion_h'),
    ('brecha_velocidad_pct','reclamaciones_30d'),
    ('reclamaciones_30d','insatisfaccion'),
    ('tiempo_resolucion_h','insatisfaccion'),
    ('insatisfaccion','churn'),
    ('precio_mensual','churn'),
]
D.add_edges_from(relaciones)
print('¿DAG acíclico?:', nx.is_directed_acyclic_graph(D))

plt.figure(figsize=(12,7))
pos = nx.spring_layout(D, seed=7, k=1.2)
nx.draw(D, pos, with_labels=True, node_color='lightblue', node_size=2600, font_size=9, arrowsize=20, edge_color='gray')
plt.title('Mapa causal hipotético: de la inversión al abandono')
plt.axis('off'); plt.tight_layout()
plt.savefig(OUTPUT_DIR/'lab09_mapa_causal_dag.png', dpi=150, bbox_inches='tight')
plt.show()

## Parte E. Confusores y caminos espurios

In [ ]:
print('Hipótesis de confusor principal: inversion_red / calidad_tecnica por región.')
print('Ejemplo: interrupciones_30d y reclamaciones_30d se correlacionan, pero ambas dependen de la calidad técnica de la red.')

print('Correlación interrupciones-reclamaciones:', round(df['interrupciones_30d'].corr(df['reclamaciones_30d']), 2))
print('Correlación calidad-reclamaciones:', round(df['calidad_tecnica'].corr(df['reclamaciones_30d']), 2))
print('Correlación calidad-interrupciones:', round(df['calidad_tecnica'].corr(df['interrupciones_30d']), 2))

## Parte F. Validación de hipótesis causales

Completa esta tabla para el entregable. Las propuestas son ejemplos editables.

In [ ]:
validacion = pd.DataFrame([
    {'hipotesis':'Invertir en red mejora calidad_tecnica', 'validacion':'cuasi-experimento antes/después por zonas con inversión'},
    {'hipotesis':'Interrupciones causan reclamaciones', 'validacion':'análisis temporal: comprobar que la interrupción precede a la reclamación'},
    {'hipotesis':'Reducir tiempo_resolucion_h baja insatisfaccion', 'validacion':'experimento A/B reforzando soporte en un grupo comparable'},
    {'hipotesis':'Precio mensual causa churn', 'validacion':'experimento de precios controlado por segmento y región'},
])
display(validacion)
validacion.to_csv(OUTPUT_DIR/'lab09_validacion_hipotesis.csv', index=False)

## Cierre

El mapa causal no demuestra causalidad por sí solo: explicita hipótesis y ayuda a decidir qué controlar y qué validar experimentalmente.